# PoliMillionaire - TF-IDF retrieval, no LLM

Runs all competitions through the API using the local SimpleWiki TF-IDF index. Each competition is played 5 times and every question attempt is printed and saved to CSV.

In [1]:
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "api_client" / "NLP_assignment_api_client").exists() and (path / "project" / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root")

PROJECT_ROOT = find_project_root()
CLIENT_PARENT = PROJECT_ROOT / "api_client" / "NLP_assignment_api_client"
SRC_DIR = PROJECT_ROOT / "project" / "src"

for path in [str(CLIENT_PARENT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\Tommaso\Code\NLP_polimi_26


In [2]:
from getpass import getpass
from millionaire_client import MillionaireClient
from retrieval_quiz_runner import load_retrieval_index, run_all_competitions, summarize, write_logs

API_URL = os.getenv("POLIMILLIONAIRE_API_URL", "http://131.175.15.22:51111/")
USERNAME = os.getenv("POLIMILLIONAIRE_USERNAME") or input("Username: ").strip()
PASSWORD = os.getenv("POLIMILLIONAIRE_PASSWORD") or getpass("Password: ")

client = MillionaireClient(API_URL, timeout=30)
user = client.login(USERNAME, PASSWORD)
print(f"Logged in as {user.username} ({user.role})")

Logged in as NeuroniNegroni (student)


In [3]:
INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / "simplewiki_160w_tfidf.joblib"
if not INDEX_PATH.exists():
    raise FileNotFoundError(f"Missing index: {INDEX_PATH}")

index = load_retrieval_index(INDEX_PATH)
print("Loaded index:", INDEX_PATH)
print("Kind:", index["kind"])
print("Chunks:", len(index["docs"]))

Loaded index: C:\Users\Tommaso\Code\NLP_polimi_26\data\indexes\simplewiki_160w_tfidf.joblib
Kind: tfidf
Chunks: 434093


In [4]:
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"{comp.id}: {comp.name} | max levels: {comp.max_levels} | questions: {comp.question_count}")

0: Entertainment | max levels: 15 | questions: 843
1: Ancient History and Politics | max levels: 15 | questions: 456
2: Science and Nature | max levels: 15 | questions: 5395
3: Maths | max levels: 15 | questions: 390


In [5]:
ATTEMPTS_PER_COMPETITION = 5
STRATEGY_NAME = "simplewiki_tfidf_no_llm"

rows = run_all_competitions(
    client=client,
    index=index,
    strategy_name=STRATEGY_NAME,
    attempts_per_competition=ATTEMPTS_PER_COMPETITION,
)

output_path = PROJECT_ROOT / "logs" / "simplewiki_tfidf_no_llm_all_competitions.csv"
write_logs(rows, output_path)
print(f"Wrote {len(rows)} rows to {output_path}")

[simplewiki_tfidf_no_llm] comp=0 Entertainment attempt=1 q=1 level=0 chosen=3 correct=False earned=0 latency=0.5979770000121789
Q: What is the fundamental principle behind Humphrey Bogart's acting career, according to his own words?
A: Always taking acting lessons
Top evidence: List of heads of state of Balochistan :: Hugh Barnes * 1899: Henry Wylie (acting) * 1899–1900: Hugh Barnes * 1900–1904: Charles Yate * 1904–1905: John Ramsay (acting) * 1905–1907: Alexander Tucker (acting) * 1907–1909: Sir Henry McMahon * 1909: Charles Archer (acting) * 1909–1911: Sir Henry McMahon * 1911–1912: John Ramsay * 1912: Charles Archer (acting) * 1912–1914: John Ramsay * 1914...

[simplewiki_tfidf_no_llm] comp=0 Entertainment attempt=2 q=1 level=0 chosen=2 correct=True earned=100 latency=0.7889100999891525
Q: Which of the following best describes the film 'Schindler's List'?
A: A historical drama about a German industrialist who saved Jews during World War II
Top evidence: Schindler's List :: Schindler

In [6]:
summarize(rows)


Summary
0 Entertainment: rows=7, sessions=5, correct=2, row_acc=0.286, max_seen_level=0, avg_latency=0.629s
1 Ancient History and Politics: rows=11, sessions=5, correct=6, row_acc=0.545, max_seen_level=0, avg_latency=0.634s
2 Science and Nature: rows=10, sessions=5, correct=5, row_acc=0.500, max_seen_level=0, avg_latency=0.680s
3 Maths: rows=8, sessions=5, correct=3, row_acc=0.375, max_seen_level=0, avg_latency=0.689s
